In [2]:
import torch
import torch.nn as nn
import random

In [4]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, embed_dim)

        self.rnn = nn.LSTM(embed_dim, self.hid_dim, n_layers, dropout=dropout)
        self.dropout =nn.Dropout(dropout)
    
    def forward(self, src):
        embedded_src = self.dropout(self.embedding(src))

        _, (hidden, cell) = self.rnn(embedded_src)

        return hidden, cell

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hidden_dim
        self.n_layers = n_layers
        
        self.embedding = nn.Embedding(output_dim, embed_dim)

        self.rnn = nn.LSTM(embed_dim, self.hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(self.hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)

        embedded_input = self.dropout(self.embedding(input))

        output, (hidden, cell) = self.rnn(embedded_input, hidden, cell)

        preds = self.fc_out(output.squeeze(0))

        return preds, hidden, cell

In [8]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, "Same Hidden dimensions must be present"
        assert encoder.n_layers == decoder.n_layers
    
    def forward(self, src, trg, teacher_forcing=0.5):

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]

        trg_vocab_size = self.decoder.output_dim

        out = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            out[t] = output
            teacher_force = random.random() < teacher_forcing 
            top1 = output.argmax(1)

            input = trg[t] if teacher_force else top1

        return out
        


In [9]:
## train